# Gen 1 / OS 1 repaired-PID field test

Copy this template to a new local notebook; preserve any existing results. This develop candidate includes the common PID source repairs. The accepted main and original BIN are unchanged. Timing findings remain open; this is a bench candidate, not timing sign-off.

## 1. Select the board and image

Edit the hostname. This cell only checks which local PyRPL installation and fork bitstream will be used.

In [ ]:
import json
from pathlib import Path
import sys

import pyrpl
from pyrpl.redpitaya import RedPitaya
from pyrpl.z10_repaired import repaired_file

HOSTNAME = "rp-xxxxxx.local"
SSH_USER = "root"
bitstream = repaired_file()

print("Python:", sys.executable)
print("PyRPL:", pyrpl.__file__)
print("Bitstream:", bitstream)
assert bitstream.is_file()

## 2. Run the read-only preflight

Enter the SSH password. This checks the OS1 ecosystem release, EEPROM board identity, character-device loader, and candidate hash without uploading or programming anything.

In [ ]:
from getpass import getpass

SSH_PASSWORD = getpass("SSH password: ")
device = None
try:
    device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=SSH_PASSWORD,
        timeout=10,
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    preflight_report = device.preflight_fpga_update(filename=str(bitstream))
finally:
    if device is not None:
        device.end_ssh()

print(json.dumps(preflight_report, indent=2, sort_keys=True))

## 3. Program the FPGA

This uploads the separate repaired Z7010 BIN and programs it through /dev/xdevcfg. No DTBO is used. It does not start the PyRPL server. Use only on the intended original Z7010 board.

In [ ]:
device = None
try:
    device = RedPitaya(
        config=None,
        hostname=HOSTNAME,
        user=SSH_USER,
        password=SSH_PASSWORD,
        filename=str(bitstream),
        timeout=10,
        gui=False,
        autostart=False,
        reloadfpga=False,
        reloadserver=False,
    )
    program_report = device.update_fpga(filename=str(bitstream))
finally:
    if device is not None:
        device.end_ssh()

print(json.dumps(program_report, indent=2, sort_keys=True))

## 4. Start PyRPL and connect

This installs/starts the monitor server. The next cell checks the fork's PID metadata. It does not reprogram the FPGA.

In [ ]:
from pyrpl import Pyrpl

p = Pyrpl(
    config="gen1-os1-repaired-field-test",
    hostname=HOSTNAME,
    user=SSH_USER,
    password=SSH_PASSWORD,
    filename=str(bitstream),
    gui=False,
    reloadfpga=False,
    reloadserver=True,
)
rp = p.rp
print("PyRPL connected; run the next cell to check PID metadata.")

## 5. Read the fork's PID metadata

Reads hardware constants and sequence state. The author's RTL uses PSR=12, ISR=32, GAINBITS=30; this is a useful comparison, not a complete fingerprint or an analog test.

In [ ]:
pid = rp.pid0
print("PID0 input filter:", pid.inputfilter)
pid_metadata = {
    "PSR": int(pid._read(0x200)),
    "ISR": int(pid._read(0x204)),
    "GAINBITS": int(pid._read(0x20C)),
    "sequence_index": int(pid.setpoint_index),
    "sequence_setpoint": float(pid.setpoint_in_sequence),
}
print(json.dumps(pid_metadata, indent=2))
assert (pid_metadata["PSR"], pid_metadata["ISR"], pid_metadata["GAINBITS"]) == (12, 32, 30)

## 6. Record the bench setup

Use OUT1 → IN1 and the external Rigol as in the Gen 1 run. Record termination, input jumper, probe and instrument settings below. Do not assume digital units equal measured volts. Keep ADC readings unclipped. This cell only defines local settings and an optional scope helper.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

TEST_CHANNEL = 1
OUTPUT_LOAD = "Hi-Z"  # Record the actual termination; this does not configure hardware.
BENCH_NOTES = ""  # Meter/scope, input jumper, cabling, measured amplitude/offset.
asg = rp.asg0
scope = rp.scope
physical_input = f"in{TEST_CHANNEL}"
physical_output = f"out{TEST_CHANNEL}"

def capture_scope(input1, input2, duration=0.02):
    scope.setup(input1=input1, input2=input2, duration=duration,
                trigger_source="immediately", trigger_delay=0,
                rolling_mode=False, average=False,
                ch1_active=True, ch2_active=True)
    try:
        data = np.asarray(scope.curve(timeout=5))
        times = scope.times.copy()
    finally:
        scope.stop()
    fig, ax = plt.subplots()
    ax.plot(times, data[0], label=input1)
    ax.plot(times, data[1], label=input2)
    ax.set(xlabel="Time [s]", ylabel="PyRPL signal units")
    ax.legend()
    plt.show()
    print("Channel means:", np.mean(data, axis=1))
    print("Channel peak-to-peak:", np.ptp(data, axis=1))
    return times, data

## 7. Disconnect fast-output routes

This changes routing for all PyRPL modules with direct fast outputs, isolating the upcoming bench signal. It does not program the FPGA. Outputs are not suitable for a connected experimental plant during these tests.

In [ ]:
for name in ("asg0", "asg1", "pid0", "pid1", "pid2",
             "iq0", "iq1", "iq2", "trig0", "trig1"):
    getattr(rp, name).output_direct = "off"
print("Fast-output routes disconnected.")

## 8. Test 1: configure the DC source

PID0 drives OUT1 with P=I=0 and adjustable integrator memory. PID1 reads IN1 without driving an output; this is the same DC-source arrangement used on Gen 1.

In [ ]:
src = rp.pid0
src.output_direct = "off"
src.input = "in1"
src.p = 0
src.i = 0
src.setpoint = 0
src.ival = 0
src.min_voltage = -0.99
src.max_voltage = 0.99
src.inputfilter = [0, 0, 0]
src.use_setpoint_sequence = False
src.paused = False

mon = rp.pid1
mon.input = "in1"
mon.output_direct = "off"
mon.p = 1
mon.i = 0
mon.setpoint = 0
mon.ival = 0
mon.min_voltage = -0.99
mon.max_voltage = 0.99
mon.inputfilter = [0, 0, 0]
mon.use_setpoint_sequence = False
mon.paused = False
src.output_direct = "out1"

## 9. Measure the five DC levels

At each of the same five commands, read the Rigol DC voltage and enter it in volts. The table preserves both RP readbacks and the physical measurement; do not enter the old Gen 1 values. On completion or interruption, the DC output is disconnected.

In [ ]:
DC_COMMANDS = [-0.5, -0.25, 0.0, 0.25, 0.5]
dc_rows = []
src.output_direct = "out1"
try:
    for command in DC_COMMANDS:
        src.ival = command
        time.sleep(0.3)
        rigol_volts = float(input(f"Command {command:+.2f}: Rigol DC voltage [V]: "))
        if not np.isfinite(rigol_volts):
            raise ValueError("Enter a finite measured voltage.")
        row = {
            "command": command,
            "rp_output": float(src.current_output_signal),
            "rp_in1": float(rp.scope.voltage_in1),
            "rp_pid1": float(mon.current_output_signal),
            "rigol_volts": rigol_volts,
        }
        dc_rows.append(row)
        print(json.dumps(row))
finally:
    src.ival = 0
    src.output_direct = "off"
print(json.dumps(dc_rows, indent=2))

## 10. Fit this board's output and input separately

Calculate volts = gain × RP readback + offset for OUT1 and IN1, with RMS residuals. These are notebook-only conversions; no driver calibration changes. The PID setpoint uses the IN1 fit, not the output fit.

In [ ]:
def fit_dc_calibration(rows):
    data = np.asarray([[r["rp_output"], r["rp_in1"], r["rigol_volts"]]
                       for r in rows], dtype=float)
    if data.ndim != 2 or data.shape[0] < 3 or data.shape[1] != 3:
        raise ValueError("At least three paired DC measurements are needed.")
    if not np.isfinite(data).all():
        raise ValueError("DC measurements must be finite.")
    fits = []
    for column in (0, 1):
        x, y = data[:, column], data[:, 2]
        if np.ptp(x) == 0:
            raise ValueError("Readback did not change across DC levels; inspect wiring/range.")
        gain, offset = np.polyfit(x, y, 1)
        if np.isclose(gain, 0):
            raise ValueError("Measured gain is zero; inspect measurements before inversion.")
        rms = float(np.sqrt(np.mean((y - (gain * x + offset)) ** 2)))
        fits.append((float(gain), float(offset), rms))
    return fits

out_fit, in_fit = fit_dc_calibration(dc_rows)
RP_OUT_TO_RIGOL_GAIN, RP_OUT_TO_RIGOL_OFFSET, out_rms = out_fit
RP_IN1_TO_RIGOL_GAIN, RP_IN1_TO_RIGOL_OFFSET, in_rms = in_fit

def rp_to_rigol(value):
    return RP_OUT_TO_RIGOL_GAIN * value + RP_OUT_TO_RIGOL_OFFSET

def rigol_to_rp(value):
    return (value - RP_OUT_TO_RIGOL_OFFSET) / RP_OUT_TO_RIGOL_GAIN

def rp_in1_to_rigol(value):
    return RP_IN1_TO_RIGOL_GAIN * value + RP_IN1_TO_RIGOL_OFFSET

def rigol_to_rp_in1(value):
    return (value - RP_IN1_TO_RIGOL_OFFSET) / RP_IN1_TO_RIGOL_GAIN

print("OUT1 fit (gain, offset V, RMS V):", out_fit)
print("IN1  fit (gain, offset V, RMS V):", in_fit)

## 11. Test 2a: 1 Hz triangle

The user confirmed both 1 Hz and 1 kHz worked on Gen 1. Start with its saved 1 Hz / amplitude 0.4 setting, then repeat at 1 kHz in step 13b. PID0 is disconnected; ASG0 drives OUT1. Record Rigol frequency, minimum, maximum, Vpp and RMS.

In [ ]:
rp.pid0.output_direct = "off"
asg = rp.asg0
asg.output_direct = "off"
asg.waveform = "ramp"
asg.frequency = 1.0  # Hz, matching the saved Gen 1 code.
asg.amplitude = 0.4
asg.offset = 0.0
asg.trigger_source = "immediately"
asg.output_direct = "out1"
print("ASG frequency [Hz]:", asg.frequency)
print("ASG amplitude [RP units]:", asg.amplitude)

## 12. Record the input trace

Collect the same 5,000 software-polled IN1 readings. Record elapsed time as well: sleep plus SSH overhead is not a uniform sample interval. Compare amplitude/offset with the Rigol; the polled mean alone is not a DC calibration.

In [ ]:
vals, sample_times = [], []
start = time.perf_counter()
for _ in range(5000):
    vals.append(rp.scope.voltage_in1)
    sample_times.append(time.perf_counter() - start)
    time.sleep(0.0005)
vals = np.asarray(vals)
sample_times = np.asarray(sample_times)
print("RP mean =", np.mean(vals))
print("RP min =", np.min(vals))
print("RP max =", np.max(vals))
print("RP Vpp =", np.ptp(vals))
print("Elapsed [s] =", sample_times[-1])

## 13. Plot and note the external measurement

Plot the captured triangle against actual read times. Enter the Rigol observations below the plot so the local notebook retains the external evidence too.

In [ ]:
fig, ax = plt.subplots()
ax.plot(sample_times, vals)
ax.set(xlabel="Elapsed [s]", ylabel="IN1 [PyRPL units]")
plt.show()
triangle_observation = input("Rigol frequency / Min / Max / Vpp / RMS and observations: ")
print("Triangle observation:", triangle_observation)

## 13b. Test 2b: 1 kHz triangle

Repeat the same amplitude at 1 kHz. Use hardware-buffered RP scope acquisition plus the Rigol, not the slow polled trace above. After recording the observation, this cell restores 1 Hz for the saved Gen 1 PID-disturbance condition.

In [ ]:
asg.frequency = 1000.0  # Hz
try:
    print("ASG frequency [Hz]:", asg.frequency)
    fast_times, fast_data = capture_scope("in1", "asg0", duration=0.01)
    fast_triangle_observation = input("Rigol 1 kHz frequency / Min / Max / Vpp / RMS and observations: ")
    print("1 kHz triangle observation:", fast_triangle_observation)
finally:
    asg.frequency = 1.0  # Restore the saved Gen 1 PID disturbance setting.

## 14. Test 3: negative-I feedback with the ASG disturbance

Repeat P=0, I=-150000 and a 0.5 V target using this board's IN1 calibration. As in the saved Gen 1 run, ASG0 stays routed to OUT1, summing its triangle with PID0 as a disturbance. The printed corrected setpoint is a calculation, not the measured output.

In [ ]:
pid = rp.pid0
pid.output_direct = "off"
pid.input = "in1"
pid.min_voltage, pid.max_voltage = -1, 1
pid.pause_gains = "pi"
pid.paused = False
pid.use_setpoint_sequence = False
pid.p = 0
pid.i = -150000
pid.ival = 0
desired_scope_voltage = 0.5
pid.setpoint = rigol_to_rp_in1(desired_scope_voltage)
pid.output_direct = "out1"
print("raw RP setpoint =", pid.setpoint)
print("scope-corrected setpoint =", rp_in1_to_rigol(pid.setpoint))
print("ASG disturbance route:", asg.output_direct)

## 15. Record the physical feedback result

Observe the Rigol before continuing: mean voltage, remaining ripple, clipping/oscillation and stability. This records the measurement separately from the calculated setpoint.

In [ ]:
pid_observation = input("Rigol PID mean / ripple / stability and observations: ")
print("PID observation:", pid_observation)
print("RP IN1 readback =", rp.scope.voltage_in1)

## 16. Turn off direct outputs

Disconnect direct outputs as in the Gen 1 notebook. Missing modules are skipped explicitly; the FPGA remains programmed.

In [ ]:
for name in ("asg0", "asg1", "pid0", "pid1", "pid2", "iq0", "iq1", "iq2", "iir", "trig0", "trig1"):
    module = getattr(rp, name, None)
    if module is None:
        print(name, "not present")
        continue
    module.output_direct = "off"
    print(name, "-> off")

## 17. Optional reconnect

Only if you want a separate reconnection check after the signal tests: close this PyRPL instance and reconnect without programming. Saved module settings are reapplied.

In [ ]:
p._clear()
p = Pyrpl(config="gen1-os1-repaired-field-test", hostname=HOSTNAME,
          user=SSH_USER, password=SSH_PASSWORD,
          filename=str(bitstream), gui=False,
          reloadfpga=False, reloadserver=False)
rp = p.rp
print("Reconnected; PID0 input filter:", rp.pid0.inputfilter)

## Results

Save this local notebook with the paired DC table, the new fits, the triangle trace and the Rigol observations. Passing this limited repeat is not a timing-closure result. TTL/hold/sequence, slow analog and bandwidth measurements are separate tests.